In [ ]:
import os
import sys
import time
from pathlib import Path
import pandas as pd
import numpy as np

# Ensure project root is in sys.path (works from tests/ or project root)
curr = os.path.abspath(os.getcwd())
while curr and curr != os.path.dirname(curr):
    if os.path.isdir(os.path.join(curr, "rfbdd")) and os.path.isdir(os.path.join(curr, "datasets")):
        project_root = curr
        break
    curr = os.path.dirname(curr)
else:
    project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from rfbdd import (
    load_forest,
    load_dataset,
    discretize,
    run_pipeline,
    SELECTORS,
)


# CONFIGURATION 
budgets = [50, 100]
timeout_seconds = 5.0      # Maximum timeout in seconds per run
max_leaves = 300           # Maximum leaf node threshold before cutoff

datasets_root = os.path.join(project_root, "datasets")

# Automatically discover all 54 pre-tuned benchmark datasets
DATASETS = sorted([
    d for d in os.listdir(datasets_root)
    if os.path.isdir(os.path.join(datasets_root, d))
    and os.path.isfile(os.path.join(datasets_root, d, f"{d}_rf.pkl"))
])

print(f"Found {len(DATASETS)} benchmark datasets: {DATASETS[:5]}... and {len(DATASETS)-5} more.")
print(f"Target Budgets: {budgets}")
print(f"Available Selectors: {list(SELECTORS.keys())}")


def execute_single_run(dataset_name: str, forest, k: int, budget: int, selector_name: str, csv_path: str, timeout=timeout_seconds, leaf_cutoff=max_leaves):
    """
    Execution of a single run for a specific dataset, selector, k, and budget
    """
    res = run_pipeline(
        dataset=forest,
        max_trees=k,
        budget=budget,
        selector=selector_name,
        timeout_seconds=timeout,
        max_leaves=leaf_cutoff,
        compute_readouts=False,
        compute_rankings=False,
        evaluate_robustness=False,
        output_dir=None,
        verbose=False,
    )
    # get stats object from the result
    st = res.stats 
    # check if the run completed successfully or timed out
    completed = st.completed if st else False
    
    row = {
        "Dataset": dataset_name,
        "Total_Trees": forest.D,
        "Features": forest.n_features,
        "Classes": forest.n_classes,
        "Trees_k": k,
        "Budget": budget,
        "Selector": selector_name,
        "Completed": completed,
        "Peak_Acc_Size": st.peak_acc_size if st else 0,
        "Peak_Manager_Size": st.peak_manager_size if st else 0,
        "Leaves": st.num_leaves if st else 0,
        "Splits": st.num_splits if st else 0,
        "Runtime_s": round(st.runtime, 4) if st else 0.0,
    }
    
    # write to the dedicated CSV for this dataset
    df_row = pd.DataFrame([row])
    write_header = not os.path.exists(csv_path) or os.path.getsize(csv_path) == 0
    try:
        df_row.to_csv(csv_path, mode="a", header=write_header, index=False)
    except PermissionError:
        fallback = csv_path.replace(".csv", "_backup.csv")
        df_row.to_csv(fallback, mode="a", header=write_header, index=False)
        
    return completed, st

def run_k_search_for_selector(dataset_name: str, forest, selector_name: str, budget: int, csv_path: str, timeout=timeout_seconds, leaf_cutoff=max_leaves):
    """Progressive search pushing k from 1 upward until timeout for a specific dataset and budget."""
    k = 1
    while k <= forest.D: # increment k until we reach the full forest size (or timeout)
        completed, st = execute_single_run(
            dataset_name=dataset_name,
            forest=forest,
            k=k,
            budget=budget,
            selector_name=selector_name,
            csv_path=csv_path,
            timeout=timeout,
            leaf_cutoff=leaf_cutoff,
        )
        if completed:
            print(f"    k={k:2d}/{forest.D:2d} -> COMPLETED | Leaves={st.num_leaves:3d} | Splits={st.num_splits:3d} | PeakAcc={st.peak_acc_size:4d} | Time={st.runtime:.3f}s")
            k += 1
        else:
            print(f"    k={k:2d}/{forest.D:2d} -> TIMEOUT/LIMIT REACHED ({st.runtime:.2f}s) -> Scaling wall found at k_max={k-1}!")
            break

def evaluate_selector_all_datasets(selector_name: str, datasets=DATASETS, budgets=budgets, timeout=timeout_seconds, leaf_cutoff=max_leaves):
    """Automates evaluation of a selector across all datasets and budgets, saving to individual dataset CSVs."""
    print("=" * 90)
    print(f" STARTING EVALUATION FOR SELECTOR: '{selector_name.upper()}' ACROSS ALL {len(datasets)} DATASETS")
    print(f" Budgets: {budgets} | Timeout: {timeout}s | Max Leaves: {leaf_cutoff}")
    print("=" * 90)
    
    for idx, d_name in enumerate(datasets, 1):
        pkl_path = os.path.join(datasets_root, d_name, f"{d_name}_rf.pkl")
        if not os.path.isfile(pkl_path):
            print(f"[!] Skipping {d_name}: forest pickle not found at {pkl_path}")
            continue
            
        forest = load_forest(pkl_path)
        
        # Setup separate CSV for this specific dataset
        dataset_res_dir = os.path.join(project_root, "results", d_name)
        os.makedirs(dataset_res_dir, exist_ok=True)
        csv_path = os.path.join(dataset_res_dir, f"{d_name}_results.csv")
        
        print(f"\n[{idx}/{len(datasets)}] DATASET: {d_name.upper()} (D={forest.D}, F={forest.n_features}, C={forest.n_classes})")
        print(f"     Saving results to: {csv_path}")
        
        for b in budgets:
            # try with full forest first; if it fails, do progressive k-search
            print(f"  -> Budget B = {b}: Testing full forest (k={forest.D})...")
            completed, st = execute_single_run(
                dataset_name=d_name,
                forest=forest,
                k=forest.D,
                budget=b,
                selector_name=selector_name,
                csv_path=csv_path,
                timeout=timeout,
                leaf_cutoff=leaf_cutoff,
            )
            
            if completed:
                print(f"     [COMPLETED] Full forest finished with D={forest.D} trees at B={b} in {st.runtime:.3f}s!")
                print(f"     No progressive k-search needed for B={b}.")
            else:
                print(f"     [TIMEOUT/LIMIT REACHED] Full D={forest.D} exceeded limits ({st.runtime:.2f}s).")
                print(f"     Starting progressive k-search (k=1..{forest.D}) for B={b}...")
                run_k_search_for_selector(
                    dataset_name=d_name,
                    forest=forest,
                    selector_name=selector_name,
                    budget=b,
                    csv_path=csv_path,
                    timeout=timeout,
                    leaf_cutoff=leaf_cutoff,
                )
    print("\n" + "=" * 90)
    print(f" FINISHED EVALUATION FOR SELECTOR: '{selector_name.upper()}' ACROSS ALL DATASETS")
    print("=" * 90)


Found 54 benchmark datasets: ['ann-thyroid', 'appendicitis', 'banknote', 'biodegradation', 'breast-cancer']... and 49 more.
Target Budgets: [50, 100]
Available Selectors: ['info_gain', 'shapley', 'p_value', 'random_node', 'modal_variable']


In [2]:
# ==============================================================================
# Selector: modal_variable (Structural mode, argmax node frequency)
# Evaluates across all 54 datasets on budgets [50, 100]
# ==============================================================================
evaluate_selector_all_datasets("modal_variable")


 STARTING EVALUATION FOR SELECTOR: 'MODAL_VARIABLE' ACROSS ALL 54 DATASETS
 Budgets: [50, 100] | Timeout: 5.0s | Max Leaves: 300

[1/54] DATASET: ANN-THYROID (D=100, F=21, C=3)
     Saving results to: c:\Users\rikyf\OneDrive\Desktop\rf-obdd-robustness\results\ann-thyroid\ann-thyroid_results.csv
  -> Budget B = 50: Testing full forest (k=100)...
     [TIMEOUT/LIMIT REACHED] Full D=100 exceeded limits (5.00s).
     Starting progressive k-search (k=1..100) for B=50...
    k= 1/100 -> COMPLETED | Leaves=  1 | Splits=  0 | PeakAcc=  64 | Time=0.011s
    k= 2/100 -> COMPLETED | Leaves= 11 | Splits= 10 | PeakAcc= 193 | Time=0.105s
    k= 3/100 -> COMPLETED | Leaves=294 | Splits=293 | PeakAcc= 472 | Time=3.404s
    k= 4/100 -> TIMEOUT/LIMIT REACHED (2.75s) -> Scaling wall found at k_max=3!
  -> Budget B = 100: Testing full forest (k=100)...
     [TIMEOUT/LIMIT REACHED] Full D=100 exceeded limits (5.00s).
     Starting progressive k-search (k=1..100) for B=100...
    k= 1/100 -> COMPLETED | Lea

In [3]:
# ==============================================================================
# Selector: random_node (Structural baseline, node-weighted random draw)
# Evaluates across all 54 datasets on budgets [50, 100]
# ==============================================================================
evaluate_selector_all_datasets("random_node")


 STARTING EVALUATION FOR SELECTOR: 'RANDOM_NODE' ACROSS ALL 54 DATASETS
 Budgets: [50, 100] | Timeout: 5.0s | Max Leaves: 300

[1/54] DATASET: ANN-THYROID (D=100, F=21, C=3)
     Saving results to: c:\Users\rikyf\OneDrive\Desktop\rf-obdd-robustness\results\ann-thyroid\ann-thyroid_results.csv
  -> Budget B = 50: Testing full forest (k=100)...
     [TIMEOUT/LIMIT REACHED] Full D=100 exceeded limits (5.00s).
     Starting progressive k-search (k=1..100) for B=50...
    k= 1/100 -> COMPLETED | Leaves=  1 | Splits=  0 | PeakAcc=  64 | Time=0.007s
    k= 2/100 -> COMPLETED | Leaves= 32 | Splits= 31 | PeakAcc= 264 | Time=0.315s
    k= 3/100 -> TIMEOUT/LIMIT REACHED (2.82s) -> Scaling wall found at k_max=2!
  -> Budget B = 100: Testing full forest (k=100)...
     [TIMEOUT/LIMIT REACHED] Full D=100 exceeded limits (5.00s).
     Starting progressive k-search (k=1..100) for B=100...
    k= 1/100 -> COMPLETED | Leaves=  1 | Splits=  0 | PeakAcc=  64 | Time=0.009s
    k= 2/100 -> COMPLETED | Leaves

In [ ]:
# ==============================================================================
# Selector: info_gain (Semantic Shannon entropy on solution set)
# Evaluates across all 54 datasets on budgets [50, 100]
# ==============================================================================
evaluate_selector_all_datasets("info_gain")

 STARTING EVALUATION FOR SELECTOR: 'INFO_GAIN' ACROSS ALL 54 DATASETS
 Budgets: [50, 100] | Timeout: 5.0s | Max Leaves: 300

[1/54] DATASET: ANN-THYROID (D=100, F=21, C=3)
     Saving results to: c:\Users\rikyf\OneDrive\Desktop\rf-obdd-robustness\results\ann-thyroid\ann-thyroid_results.csv
  -> Budget B = 50: Testing full forest (k=100)...
     [TIMEOUT/LIMIT REACHED] Full D=100 exceeded limits (5.00s).
     Starting progressive k-search (k=1..100) for B=50...
    k= 1/100 -> COMPLETED | Leaves=  1 | Splits=  0 | PeakAcc=  64 | Time=0.013s
    k= 2/100 -> COMPLETED | Leaves= 11 | Splits= 10 | PeakAcc= 354 | Time=0.357s
    k= 3/100 -> TIMEOUT/LIMIT REACHED (3.63s) -> Scaling wall found at k_max=2!
  -> Budget B = 100: Testing full forest (k=100)...
     [TIMEOUT/LIMIT REACHED] Full D=100 exceeded limits (5.00s).
     Starting progressive k-search (k=1..100) for B=100...
    k= 1/100 -> COMPLETED | Leaves=  1 | Splits=  0 | PeakAcc=  64 | Time=0.008s
    k= 2/100 -> COMPLETED | Leaves= 

In [5]:
# ==============================================================================
# Selector: shapley (Semantic game-theoretic coalition value)
# Evaluates across all 54 datasets on budgets [50, 100]
# ==============================================================================
evaluate_selector_all_datasets("shapley")


 STARTING EVALUATION FOR SELECTOR: 'SHAPLEY' ACROSS ALL 54 DATASETS
 Budgets: [50, 100] | Timeout: 5.0s | Max Leaves: 300

[1/54] DATASET: ANN-THYROID (D=100, F=21, C=3)
     Saving results to: c:\Users\rikyf\OneDrive\Desktop\rf-obdd-robustness\results\ann-thyroid\ann-thyroid_results.csv
  -> Budget B = 50: Testing full forest (k=100)...
     [TIMEOUT/LIMIT REACHED] Full D=100 exceeded limits (5.00s).
     Starting progressive k-search (k=1..100) for B=50...
    k= 1/100 -> COMPLETED | Leaves=  1 | Splits=  0 | PeakAcc=  64 | Time=0.013s
    k= 2/100 -> COMPLETED | Leaves= 11 | Splits= 10 | PeakAcc= 354 | Time=0.307s
    k= 3/100 -> TIMEOUT/LIMIT REACHED (4.33s) -> Scaling wall found at k_max=2!
  -> Budget B = 100: Testing full forest (k=100)...
     [TIMEOUT/LIMIT REACHED] Full D=100 exceeded limits (5.00s).
     Starting progressive k-search (k=1..100) for B=100...
    k= 1/100 -> COMPLETED | Leaves=  1 | Splits=  0 | PeakAcc=  64 | Time=0.007s
    k= 2/100 -> COMPLETED | Leaves=  2

In [ ]:
# ==============================================================================
# Selector: p_value (Semantic hypergeometric association rule tail)
# Evaluates across all 54 datasets on budgets [50, 100]
# ==============================================================================
evaluate_selector_all_datasets("p_value")


 STARTING EVALUATION FOR SELECTOR: 'P_VALUE' ACROSS ALL 54 DATASETS
 Budgets: [50, 100] | Timeout: 5.0s | Max Leaves: 300

[1/54] DATASET: ANN-THYROID (D=100, F=21, C=3)
     Saving results to: c:\Users\rikyf\OneDrive\Desktop\rf-obdd-robustness\results\ann-thyroid\ann-thyroid_results.csv
  -> Budget B = 50: Testing full forest (k=100)...
     [TIMEOUT/LIMIT REACHED] Full D=100 exceeded limits (5.90s).
     Starting progressive k-search (k=1..100) for B=50...
    k= 1/100 -> COMPLETED | Leaves=  1 | Splits=  0 | PeakAcc=  64 | Time=0.007s
    k= 2/100 -> COMPLETED | Leaves= 11 | Splits= 10 | PeakAcc= 354 | Time=0.237s
    k= 3/100 -> TIMEOUT/LIMIT REACHED (5.00s) -> Scaling wall found at k_max=2!
  -> Budget B = 100: Testing full forest (k=100)...
     [TIMEOUT/LIMIT REACHED] Full D=100 exceeded limits (5.14s).
     Starting progressive k-search (k=1..100) for B=100...
    k= 1/100 -> COMPLETED | Leaves=  1 | Splits=  0 | PeakAcc=  64 | Time=0.008s
    k= 2/100 -> COMPLETED | Leaves=  2

In [2]:
# ==============================================================================
# Selector: p_value (RESUME from 'ecoli' to the end of DATASETS)
# Evaluates remaining datasets from ecoli (index 10) through zernike (index 54)
# ==============================================================================
start_idx = DATASETS.index("ecoli")
print(f"Resuming p_value evaluation from dataset {start_idx + 1}/54: {DATASETS[start_idx]}")
evaluate_selector_all_datasets("p_value", datasets=DATASETS[start_idx:])


Resuming p_value evaluation from dataset 10/54: ecoli
 STARTING EVALUATION FOR SELECTOR: 'P_VALUE' ACROSS ALL 45 DATASETS
 Budgets: [50, 100] | Timeout: 5.0s | Max Leaves: 300

[1/45] DATASET: ECOLI (D=39, F=7, C=5)
     Saving results to: c:\Users\rikyf\OneDrive\Desktop\rf-obdd-robustness\results\ecoli\ecoli_results.csv
  -> Budget B = 50: Testing full forest (k=39)...
     [TIMEOUT/LIMIT REACHED] Full D=39 exceeded limits (5.00s).
     Starting progressive k-search (k=1..39) for B=50...
    k= 1/39 -> COMPLETED | Leaves=  1 | Splits=  0 | PeakAcc=  55 | Time=0.008s
    k= 2/39 -> COMPLETED | Leaves= 21 | Splits= 20 | PeakAcc= 187 | Time=0.336s
    k= 3/39 -> TIMEOUT/LIMIT REACHED (3.57s) -> Scaling wall found at k_max=2!
  -> Budget B = 100: Testing full forest (k=39)...
     [TIMEOUT/LIMIT REACHED] Full D=39 exceeded limits (5.00s).
     Starting progressive k-search (k=1..39) for B=100...
    k= 1/39 -> COMPLETED | Leaves=  1 | Splits=  0 | PeakAcc=  55 | Time=0.006s
    k= 2/39 ->

In [ ]:

# Summary: Aggregation of All 54 Dataset CSVs 
results_base_dir = os.path.join(project_root, "results")
all_csv_files = []

for d_name in DATASETS:
    c_path = os.path.join(results_base_dir, d_name, f"{d_name}_results.csv")
    if os.path.exists(c_path):
        all_csv_files.append(c_path)

print(f"Found {len(all_csv_files)} / {len(DATASETS)} dataset result CSVs.")

if all_csv_files:
    df_all = pd.concat([pd.read_csv(f) for f in all_csv_files], ignore_index=True)
    master_csv_path = os.path.join(results_base_dir, "deliverable_6_all_datasets_summary.csv")
    df_all.to_csv(master_csv_path, index=False)
    print(f"Saved master combined CSV ({len(df_all)} rows) to: {master_csv_path}\n")
    
    print("=" * 90)
    print(" COMPLETION RATE ACROSS BUDGETS AND SELECTORS")
    print("=" * 90)
    comp_summary = df_all.groupby(["Budget", "Selector"])["Completed"].agg(total_runs="count", completed_runs="sum")
    comp_summary["completion_rate"] = (comp_summary["completed_runs"] / comp_summary["total_runs"]).round(3)
    print(comp_summary)
    
    print("\n" + "=" * 90)
    print(" AVERAGE SCALING WALL (k_max) PER SELECTOR AND BUDGET")
    print("=" * 90)
    completed_rows = df_all[df_all["Completed"] == True]
    if not completed_rows.empty:
        k_max_per_dataset = completed_rows.groupby(["Dataset", "Budget", "Selector"])["Trees_k"].max().reset_index()
        avg_k_max = k_max_per_dataset.groupby(["Budget", "Selector"])["Trees_k"].agg(["mean", "median", "min", "max"]).round(2)
        print(avg_k_max)
else:
    print("No results CSVs found yet. Run the selector cells above first.")


Found 54 / 54 dataset result CSVs.
Saved master combined CSV (3338 rows) to: c:\Users\rikyf\OneDrive\Desktop\rf-obdd-robustness\results\deliverable_6_all_datasets_summary.csv

 COMPLETION RATE ACROSS BUDGETS AND SELECTORS
                       total_runs  completed_runs  completion_rate
Budget Selector                                                   
50     info_gain              314             208            0.662
       modal_variable         354             248            0.701
       p_value                300             194            0.647
       random_node            327             221            0.676
       shapley                320             214            0.669
100    info_gain              326             222            0.681
       modal_variable         394             290            0.736
       p_value                338             234            0.692
       random_node            332             230            0.693
       shapley                333        